## Import Required Libraries

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

### Load Dataset

In [8]:
df = pd.read_csv(r"D:\NewFolder_Datasets\Titanic-Dataset.csv")


### Remove Unnecessary Columns

In [9]:
df = df.drop(columns=['PassengerId','Name','Ticket','Cabin'])

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


### Handle Missing Values

In [10]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

### Convert Categorical Data into Numerical

In [11]:
df['Sex'] = df['Sex'].map({
    'male':0,
    'female':1
})

df['Embarked'] = df['Embarked'].map({
    'S':0,
    'C':1,
    'Q':2
})

### Select Features and Target

In [12]:
X = df[['Pclass','Sex','Age','Fare']]

y = df['Survived']

### Train-Test Split

In [14]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

### Feature Scaling

In [15]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

### Logistic Regression

In [16]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train,y_train)

lr_predict = lr.predict(X_test)

lr_accuracy = accuracy_score(y_test,lr_predict)

print("Logistic Regression Accuracy =",lr_accuracy)

Logistic Regression Accuracy = 0.7988826815642458


### Naive Bayes

In [17]:
nb = GaussianNB()

nb.fit(X_train,y_train)

nb_predict = nb.predict(X_test)

nb_accuracy = accuracy_score(y_test,nb_predict)

print("Naive Bayes Accuracy =",nb_accuracy)

Naive Bayes Accuracy = 0.7597765363128491


### KNN

In [18]:
knn = KNeighborsClassifier()

knn.fit(X_train,y_train)

knn_predict = knn.predict(X_test)

knn_accuracy = accuracy_score(y_test,knn_predict)

print("KNN Accuracy =",knn_accuracy)

KNN Accuracy = 0.8212290502793296


### Support Vector Machine

In [19]:
svm = SVC()

svm.fit(X_train,y_train)

svm_predict = svm.predict(X_test)

svm_accuracy = accuracy_score(y_test,svm_predict)

print("SVM Accuracy =",svm_accuracy)

SVM Accuracy = 0.7988826815642458


### Compare Accuracy

In [20]:

print("Logistic Regression :",lr_accuracy)

print("Naive Bayes :",nb_accuracy)

print("KNN :",knn_accuracy)

print("SVM :",svm_accuracy)

Logistic Regression : 0.7988826815642458
Naive Bayes : 0.7597765363128491
KNN : 0.8212290502793296
SVM : 0.7988826815642458


So,

Best Model = SVM

### Hyperparameter Tuning using GridSearchCV

In [21]:
parameter = {

    'C':[0.1,1,10,100],

    'kernel':['linear','rbf'],

    'gamma':['scale','auto']

}

In [22]:
grid = GridSearchCV(

    SVC(),

    parameter,

    cv=5,

    scoring='accuracy'

)

grid.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

### Best Parameters

In [23]:
print(grid.best_params_)

{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}


### Best Cross Validation Accuracy

In [24]:
print(grid.best_score_)

0.821609376538954


### Train Final Model

In [25]:
final_model = SVC(

    C=10,

    kernel='rbf',

    gamma='scale'

)

final_model.fit(X_train,y_train)

SVC(C=10)

### Prediction

In [26]:
prediction = final_model.predict(X_test)

### Final Accuracy

In [27]:
final_accuracy = accuracy_score(y_test,prediction)

print("Final Accuracy =",final_accuracy)

Final Accuracy = 0.8156424581005587


### Confusion Matrix

In [28]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,prediction)

print(cm)

[[95 10]
 [23 51]]


### Classification Report

In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.84      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179



Q. Why did you apply GridSearchCV?

Answer: GridSearchCV tries different combinations of hyperparameters using cross-validation and selects the combination that gives the highest validation accuracy.

Q. Why did you train the model again?

Answer: After finding the best hyperparameters from GridSearchCV, we train the model again using those optimal values to obtain the best possible performance on the test data.

This is the style most faculty expect in practical exams: each algorithm is trained separately, accuracies are printed one by one, the best model is selected manually, then GridSearchCV is applied only to that model, followed by retraining with the best parameters.

Feature Scaling (Recommended for Logistic Regression, KNN, SVM)

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Hyperparameter Tuning using GridSearchCV

Logistic Regression

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

parameter = {
    'C':[0.01,0.1,1,10,100],
    'solver':['liblinear','lbfgs']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000),
                    parameter,
                    cv=5)

grid.fit(X_train,y_train)

print(grid.best_params_)
print(grid.best_score_)

{'C': 0.1, 'solver': 'lbfgs'}
0.7920811582783414


KNN

In [32]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

parameter = {
    'n_neighbors':[3,5,7,9,11],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

grid = GridSearchCV(KNeighborsClassifier(),
                    parameter,
                    cv=5)

grid.fit(X_train,y_train)

print(grid.best_params_)

{'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}


SVM

In [33]:
parameter = {

    'C':[0.1,1,10,100],

    'kernel':['linear','rbf'],

    'gamma':['scale','auto']
}

grid = GridSearchCV(
    SVC(),
    parameter,
    cv=5
)

grid.fit(X_train,y_train)

print(grid.best_params_)

{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}


Feature Selection

Keep only the important features.

In [34]:
X = df[['Pclass',
        'Sex',
        'Age',
        'Fare']]

Remove Outliers

In [35]:
Q1 = df['Fare'].quantile(0.25)

Q3 = df['Fare'].quantile(0.75)

IQR = Q3-Q1

df = df[
    (df['Fare']>=Q1-1.5*IQR) &
    (df['Fare']<=Q3+1.5*IQR)
]

Handle Missing Values Properly

In [36]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')

X = imputer.fit_transform(X)

Cross Validation

In [37]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    SVC(),
    X,
    y,
    cv=10
)

print(scores)

print(scores.mean())

[0.57777778 0.58426966 0.68539326 0.74157303 0.6741573  0.69662921
 0.68539326 0.71910112 0.70786517 0.6741573 ]
0.6746317103620474


Feature Engineering

In [38]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

C:\Users\Gopal Chavan\AppData\Local\Temp\ipykernel_15744\4092208064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['FamilySize'] = df['SibSp'] + df['Parch'] + 1


In [39]:
df['IsAlone'] = (df['FamilySize']==1).astype(int)

C:\Users\Gopal Chavan\AppData\Local\Temp\ipykernel_15744\717368415.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['IsAlone'] = (df['FamilySize']==1).astype(int)


Use these features:

In [40]:
X = df[['Pclass',
        'Sex',
        'Age',
        'Fare',
        'FamilySize',
        'IsAlone']]

Class Weight (for Imbalanced Data)

In [41]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

Try Ensemble Models

In [42]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train,y_train)

prediction = rf.predict(X_test)

In [43]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier()

gb.fit(X_train,y_train)

GradientBoostingClassifier()

Normalize Data

In [47]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

Retrain Using Best Parameters

In [48]:
best_model = SVC(
    C=10,
    kernel='rbf',
    gamma='scale'
)

best_model.fit(X_train,y_train)

prediction = best_model.predict(X_test)